In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import numpy as np
from pyqpanda3.core import QCircuit, RZ, RX, RY
from pyqpanda_alg.QPE import QPE

# QPE Demo 2: Custom Unitary & Precision

This notebook shows QPE with rotation gates and tests
how precision improves with more counting qubits.

- RZ(θ)|0⟩ = e^(-iθ/2)|0⟩ →  φ = -θ/(4π) mod 1
- More counting qubits → higher precision

In [ ]:
# RZ(π/2): φ = -π/8 mod 1 = 0.875
def u_rz(tq):
    cir = QCircuit()
    cir << RZ(tq[0], np.pi / 2)
    return cir

qpe = QPE(unitary=u_rz, n_count=6, n_target=1)
phi = qpe.run()
print(f"RZ(π/2): φ = {phi:.4f} (expect 0.875)")
print(f"Precision: 1/2^{qpe.n_count} = {1/(1<<qpe.n_count):.4f}")

In [ ]:
# Compare precision vs counting qubits
for nc in [2, 3, 4, 5, 6]:
    q = QPE(unitary=u_rz, n_count=nc, n_target=1)
    p = q.run()
    err = abs(p - 0.875)
    print(f"  n={nc}: φ={p:.4f}  error={err:.4f}")

In [ ]:
# Multi-qubit unitary example: RX(0.3) on 2 qubits
def u_rx(tq):
    cir = QCircuit()
    cir << RX(tq[0], 0.3)
    cir << RX(tq[1], 0.3)
    return cir

# RX(θ)|0⟩ = cos(θ/2)|0⟩ - i sin(θ/2)|1⟩ →  eigenvalue info via QPE
qpe_rx = QPE(unitary=u_rx, n_count=6, n_target=2)
dist = qpe_rx.run_dict()
print("Phase distribution (2-target RX):")
for k, v in sorted(dist.items()):
    if v > 0.01:
        phi_val = int(k, 2) / (1 << 6)
        print(f"  |{k}⟩  φ={phi_val:.4f}  prob={v:.4f}")